In [1]:
import pandas as pd
from rdkit import Chem

# --- 1. Define Classification Logic ---
def classify_component(smiles):
    """
    Classifies a single molecular component.
    """
    if pd.isna(smiles) or smiles == "":
        return "Invalid", "Empty"
    
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return "Invalid", "RDKit parse error"

    # Pattern: Alpha-Beta Unsaturated Ester/Acid (C=C-C(=O)O)
    acrylate_pattern = Chem.MolFromSmarts('[CX3]=[CX3][CX3](=[OX1])[OX2]')
    
    if not mol.HasSubstructMatch(acrylate_pattern):
        return "Not Acrylate", "No acrylic moiety"

    matches = mol.GetSubstructMatches(acrylate_pattern)
    
    has_acrylate = False
    has_methacrylate = False
    has_beta_sub = False # Cinnamate-like
    has_alpha_sub = False
    
    for match in matches:
        beta_idx, alpha_idx, carbonyl_idx = match[0], match[1], match[2]
        beta_atom = mol.GetAtomWithIdx(beta_idx)
        alpha_atom = mol.GetAtomWithIdx(alpha_idx)
        
        # Check if terminal alkene (Beta has 2 Hydrogens)
        if beta_atom.GetTotalNumHs() == 2:
            # Check Alpha substitution
            if alpha_atom.GetTotalNumHs() == 1:
                has_acrylate = True # Standard Acrylate
            else:
                # Check if the substituent is Methyl (Methacrylate)
                is_methyl = False
                for neighbor in alpha_atom.GetNeighbors():
                    nid = neighbor.GetIdx()
                    if nid not in [beta_idx, carbonyl_idx]:
                        if neighbor.GetAtomicNum() == 6 and neighbor.GetTotalNumHs() == 3:
                            is_methyl = True
                
                if is_methyl:
                    has_methacrylate = True
                else:
                    has_alpha_sub = True
        else:
            has_beta_sub = True # Internal alkene (Cinnamate, etc)

    # Priority Classification
    if has_acrylate: return "Acrylate", "Terminal CH2=CH-COO"
    if has_methacrylate: return "Methacrylate", "Terminal CH2=C(Me)-COO"
    if has_alpha_sub: return "Alpha-Substituted", "Terminal CH2=C(R)-COO"
    if has_beta_sub: return "Beta-Substituted", "Internal double bond"
    
    return "Uncertain", "Pattern matched but logic unclear"

# --- 2. Load Data ---
df = pd.read_csv('acrylates.csv')

# --- 3. Separate Mixtures (The "Explode" Step) ---
# Split the SMILES string by '.' into a list of strings
df['smiles_component'] = df['smiles'].str.split('.')

# Explode the lists into separate rows
# (The original 'id' and 'cmpdname' are duplicated for each component)
df_separated = df.explode('smiles_component')

# --- 4. Classify Each Component ---
print("Classifying components...")
results = df_separated['smiles_component'].apply(classify_component)

df_separated['Category'] = [r[0] for r in results]
df_separated['Reason'] = [r[1] for r in results]

# --- 5. Save Result ---
df_separated = df_separated.drop(columns=['smiles']).rename(columns={'smiles_component':'smiles'})
output_filename = 'acrylates_classified.csv'
df_separated.to_csv(output_filename, index=False)

Classifying components...


[15:02:20] WARNING: not removing hydrogen atom without neighbors


In [2]:
df_separated

,id,cmpdname,mw,mf,polararea,complexity,smiles,Category,Reason
0,6581,Acrylic acid,72.060,C3H4O2,37.300,55.9,C=CC(=O)O,Acrylate,Terminal CH2=CH-COO
1,13165,2-Hydroxyethyl acrylate,116.110,C5H8O3,46.500,87.7,C=CC(=O)OCCO,Acrylate,Terminal CH2=CH-COO
2,5355130,Octinoxate,290.400,C18H26O3,35.500,304.0,CCCCC(CC)COC(=O)/C=C/c1ccc(OC)cc1,Beta-Substituted,Internal double bond
3,8846,Butyl Acrylate,128.169,C7H12O2,26.300,97.1,C=CC(=O)OCCCC,Acrylate,Terminal CH2=CH-COO
4,8821,Ethyl acrylate,100.120,C5H8O2,26.300,76.1,C=CC(=O)OCC,Acrylate,Terminal CH2=CH-COO
...,...,...,...,...,...,...,...,...,...
9621,134767817,"[(1S,2R,5S)-2-[2-(4-methoxyphenyl)propan-2-yl]...",316.400,C20H28O3,35.500,407.0,C=CC(=O)O[C@H]1C[C@@H](C)CC[C@@H]1C(C)(C)c1ccc...,Acrylate,Terminal CH2=CH-COO
9622,166640333,"[(E,2R)-5-oxo-5-(N-phenylanilino)pent-3-en-2-y...",461.500,C28H31NO5,65.099,711.0,C[C@H](/C=C/C(=O)N(c1ccccc1)c1ccccc1)OC(=O)/C=...,Beta-Substituted,Internal double bond
9623,789517,"ethyl (2E)-2-cyano-3-(4,5-dimethoxy-2-nitrophe...",306.270,C14H14N2O6,114.000,489.0,CCOC(=O)/C(C#N)=C/c1cc(OC)c(OC)cc1[N+](=O)[O-],Beta-Substituted,Internal double bond
9624,98473754,4-formyl-2-methoxyphenyl (2E)-3-(4-tert-butylp...,338.400,C21H22O4,52.600,469.0,COc1cc(C=O)ccc1OC(=O)/C=C/c1ccc(C(C)(C)C)cc1,Beta-Substituted,Internal double bond


In [3]:
df_separated[df_separated['Category'] == 'Acrylate'].drop(columns=['Category', 'Reason']).to_csv('../polygraphpy/data/full_dataset.csv', index=False)